In [1]:
import pandas as pd
import numpy as np
import sqlite3
import datetime
import warnings
warnings.filterwarnings('ignore')

# Extract

In [2]:
csv_path = 'airline_route_profitability.csv'
try:
    df_raw = pd.read_csv(csv_path)
    print("DATASET INFORMATION")
    df_raw_info = pd.DataFrame({
        "dtype": df_raw.dtypes.astype(str),
        "n_unique": df_raw.nunique(),
        "n_missing": df_raw.isna().sum(),
        "pct_missing": (df_raw.isna().mean() * 100).round(2),
        "example_value": [df_raw[c].dropna().iloc[0] if df_raw[c].notna().any() else None for c in df_raw.columns]
    })
    display(df_raw_info)
except FileNotFoundError:
    print("File not found")
    raise

DATASET INFORMATION


,dtype,n_unique,n_missing,pct_missing,example_value
Flight_Number,str,7974,0,0.00,EK8960
Flight_Date,str,366,0,0.00,2024-12-20
Origin,str,1,0,0.00,DXB
Destination,str,30,0,0.00,ORD
Route,str,30,0,0.00,DXB-ORD
Aircraft_Type,str,6,0,0.00,Boeing 777-300ER
Aircraft_Capacity,int64,6,0,0.00,396
Passengers,int64,388,0,0.00,308
Load_Factor,float64,2959,0,0.00,0.7791
Flight_Hours,float64,21,0,0.00,14.5


In [3]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 7974 entries, 0 to 7973
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Flight_Number            7974 non-null   str    
 1   Flight_Date              7974 non-null   str    
 2   Origin                   7974 non-null   str    
 3   Destination              7974 non-null   str    
 4   Route                    7974 non-null   str    
 5   Aircraft_Type            7974 non-null   str    
 6   Aircraft_Capacity        7974 non-null   int64  
 7   Passengers               7974 non-null   int64  
 8   Load_Factor              7974 non-null   float64
 9   Flight_Hours             7974 non-null   float64
 10  Season                   7974 non-null   str    
 11  Route_Category           7974 non-null   str    
 12  Demand_Level             7974 non-null   str    
 13  Ticket_Revenue           7974 non-null   float64
 14  Ancillary_Revenue        7703 non-n

In [4]:
# Statistik kolom numerik
df_raw.describe(include=[np.number]).T

,count,mean,std,min,25%,50%,75%,max
Aircraft_Capacity,7974.0,317.800602,98.768543,180.0000,296.000000,325.0000,396.0000,517.00
Passengers,7974.0,254.819162,88.141968,104.0000,180.250000,246.0000,302.0000,491.00
Load_Factor,7974.0,0.801570,0.087755,0.5766,0.736825,0.8015,0.8731,0.95
Flight_Hours,7974.0,6.327314,4.555668,1.2000,3.200000,4.5000,7.5000,16.50
Ticket_Revenue,7974.0,264383.873745,237102.214088,14812.3400,56429.512500,182620.3550,412696.9325,1303644.62
Ancillary_Revenue,7703.0,33073.717197,30167.518230,1616.6200,7234.405000,22842.8500,51234.9750,192921.91
Total_Revenue,7974.0,297436.197695,266832.892069,16650.0200,63502.390000,204812.5900,463800.2450,1496566.53
Fuel_Cost,7974.0,46296.405878,43123.036900,2592.6200,16137.575000,28344.9500,74832.9600,191047.78
Maintenance_Cost,7974.0,18973.224856,17677.194300,1380.0000,7040.000000,11200.0000,30800.0000,72500.00
Crew_Cost,7974.0,8905.116880,7414.772748,936.0000,3840.000000,5850.0000,13000.0000,29000.00


In [5]:
cost_components = [
    "Fuel_Cost", "Maintenance_Cost", "Crew_Cost", "Depreciation_Cost", "Insurance_Cost",
    "Airport_Fees", "Catering_Cost", "Handling_Cost", "Navigation_Fees",
    "Sales_Distribution_Cost", "Passenger_Service_Cost", "Overhead_Cost",
    "Marketing_Cost", "IT_Systems_Cost"
]

In [6]:
print("========== DATA QUALITY ==========")

print("\nMissing Values : ")
missing_values = df_raw.isnull().sum()
print(missing_values[missing_values > 0])

print("\nDuplicate Data : ")
print(df_raw.duplicated().sum())

print("\nInconsistent Total Revenue (diff > 1) : ", )
if {"Ticket_Revenue", "Ancillary_Revenue", "Total_Revenue"}.issubset(df_raw.columns):
    diff_rev = (df_raw["Total_Revenue"] - (df_raw["Ticket_Revenue"] + df_raw["Ancillary_Revenue"])).abs()
    print(int((diff_rev > 1).sum()))

print("\nInconsistent Total Cost (diff > 1) : ", )
if cost_components and "Total_Cost" in df_raw.columns:
    diff_cost = (df_raw["Total_Cost"] - df_raw[cost_components].sum(axis=1)).abs()
    print(int((diff_cost > 1).sum()))

print("\nInconsistent Profit (diff > 1) : ", )
if {"Total_Revenue", "Total_Cost", "Profit"}.issubset(df_raw.columns):
    diff_profit = (df_raw["Profit"] - (df_raw["Total_Revenue"] - df_raw["Total_Cost"])).abs()
    print(int((diff_profit > 1).sum()))

print("\nRoute-Origin-Destination Mismatch : ", )
if {"Route", "Origin", "Destination"}.issubset(df_raw.columns):
    expected_route = df_raw["Origin"].astype(str).str.strip() + "-" + df_raw["Destination"].astype(str).str.strip()
    mismatch = (df_raw["Route"].astype(str).str.strip() != expected_route).sum()
    print(int(mismatch))

========== DATA QUALITY ==========

Missing Values : 
Ancillary_Revenue    271
Catering_Cost        265
Handling_Cost        256
dtype: int64

Duplicate Data : 
0

Inconsistent Total Revenue (diff > 1) : 
0

Inconsistent Total Cost (diff > 1) : 
521

Inconsistent Profit (diff > 1) : 
0

Route-Origin-Destination Mismatch : 
0


# Transform

In [7]:
df = df_raw.copy()

In [8]:
# Lowercase all column names
df.columns = df.columns.str.lower()

print("========== Before After Lowercase Column Names ==========")
print("Before:", df_raw.columns)
print("After:", df.columns)

========== Before After Lowercase Column Names ==========
Before: Index(['Flight_Number', 'Flight_Date', 'Origin', 'Destination', 'Route',
       'Aircraft_Type', 'Aircraft_Capacity', 'Passengers', 'Load_Factor',
       'Flight_Hours', 'Season', 'Route_Category', 'Demand_Level',
       'Ticket_Revenue', 'Ancillary_Revenue', 'Total_Revenue', 'Fuel_Cost',
       'Maintenance_Cost', 'Crew_Cost', 'Depreciation_Cost', 'Insurance_Cost',
       'Airport_Fees', 'Catering_Cost', 'Handling_Cost', 'Navigation_Fees',
       'Sales_Distribution_Cost', 'Passenger_Service_Cost', 'Overhead_Cost',
       'Marketing_Cost', 'IT_Systems_Cost', 'Total_Cost', 'Profit',
       'Profit_Margin'],
      dtype='str')
After: Index(['flight_number', 'flight_date', 'origin', 'destination', 'route',
       'aircraft_type', 'aircraft_capacity', 'passengers', 'load_factor',
       'flight_hours', 'season', 'route_category', 'demand_level',
       'ticket_revenue', 'ancillary_revenue', 'total_revenue', 'fuel_cost',
   

In [9]:
# Missing Value Handling
df['ancillary_revenue'] = df['ancillary_revenue'].fillna(0)
df['catering_cost'] = df['catering_cost'].fillna(0)
df['handling_cost'] = df['handling_cost'].fillna(0)

print("========== Before After Missing Value Handling ==========")
print("Before : ")
missing_values_before = df_raw.isnull().sum()
print(missing_values_before[missing_values_before > 0] if missing_values_before[missing_values_before > 0].any() else "0")
print("\nAfter : ")
missing_values_after = df.isnull().sum()
print(missing_values_after[missing_values_after > 0] if missing_values_after[missing_values_after > 0].any() else "0")

========== Before After Missing Value Handling ==========
Before : 
Ancillary_Revenue    271
Catering_Cost        265
Handling_Cost        256
dtype: int64

After : 
0


In [10]:
# Convert date columns to datetime
df['flight_date'] = pd.to_datetime(df['flight_date'])

print("========== Before After Date Conversion ==========")
print("Before : ", df_raw['Flight_Date'].dtype)
print("After : ", df['flight_date'].dtype)

========== Before After Date Conversion ==========
Before :  str
After :  datetime64[us]


In [11]:
cost_components_lower = [col.lower() for col in cost_components]
cost_components_lower

['fuel_cost',
 'maintenance_cost',
 'crew_cost',
 'depreciation_cost',
 'insurance_cost',
 'airport_fees',
 'catering_cost',
 'handling_cost',
 'navigation_fees',
 'sales_distribution_cost',
 'passenger_service_cost',
 'overhead_cost',
 'marketing_cost',
 'it_systems_cost']

In [12]:
# Recalculate total_cost based on the sum of cost components
df['total_cost'] = df[cost_components_lower].sum(axis=1)

# Update Profit dan Profit_Margin (dihitung ulang dari total_revenue & total_cost yang sudah dikoreksi)
df['profit'] = df['total_revenue'] - df['total_cost']
# Calculate profit margin, handling division by zero
df['profit_margin'] = df.apply(
    lambda row: row['profit'] / row['total_revenue'] if row['total_revenue'] > 0 else 0, 
    axis=1
)

print("========== Before After Revenue & Cost Recalculation ==========")
print("Before : ")
print("Inconsistent Total Cost (diff > 1) : ", )
if cost_components and "Total_Cost" in df_raw.columns:
    diff_cost = (df_raw["Total_Cost"] - df_raw[cost_components].sum(axis=1)).abs()
    print(int((diff_cost > 1).sum()))

print("\nAfter : ")
print("Inconsistent Total Cost (diff > 1) : ", )
diff_cost_after = (df["total_cost"] - df[cost_components_lower].sum(axis=1)).abs()
print(int((diff_cost_after > 1).sum()))


========== Before After Revenue & Cost Recalculation ==========
Before : 
Inconsistent Total Cost (diff > 1) : 
521

After : 
Inconsistent Total Cost (diff > 1) : 
0


## _Dim & Facts_

In [13]:
# DIM_DEMAND_LEVEL
dim_demand_level = df[['demand_level']].drop_duplicates().reset_index(drop=True)
dim_demand_level['demand_level_id'] = dim_demand_level.index + 1
dim_demand_level = dim_demand_level[['demand_level_id', 'demand_level']]

print("dim_demand_level:", dim_demand_level.shape)
dim_demand_level

dim_demand_level: (2, 2)


,demand_level_id,demand_level
0,1,Medium
1,2,High


In [14]:
# DIM_SEASON
dim_season = df[['season']].drop_duplicates().reset_index(drop=True)
dim_season['season_id'] = dim_season.index + 1
dim_season = dim_season[['season_id', 'season']]

print("dim_season:", dim_season.shape)
dim_season

dim_season: (4, 2)


,season_id,season
0,1,Peak
1,2,Normal
2,3,Shoulder
3,4,Low


In [15]:
# DIM_DATE
dim_date = df[['flight_date']].drop_duplicates().reset_index(drop=True)
dim_date['date_id'] = dim_date['flight_date'].dt.strftime('%Y%m%d').astype(int)
dim_date['day'] = dim_date['flight_date'].dt.day
dim_date['month'] = dim_date['flight_date'].dt.month
dim_date['month_name'] = dim_date['flight_date'].dt.strftime('%B')
dim_date['quarter'] = dim_date['flight_date'].dt.quarter
dim_date['year'] = dim_date['flight_date'].dt.year
dim_date['is_weekend'] = dim_date['flight_date'].dt.dayofweek.isin([5, 6]).astype(int)
dim_date = dim_date[['date_id', 'flight_date', 'day', 'month', 'month_name', 'quarter', 'year', 'is_weekend']]

print("dim_date:", dim_date.shape)
dim_date.head()

dim_date: (366, 8)


,date_id,flight_date,day,month,month_name,quarter,year,is_weekend
0,20241220,2024-12-20,20,12,December,4,2024,0
1,20240513,2024-05-13,13,5,May,2,2024,0
2,20241012,2024-10-12,12,10,October,4,2024,1
3,20240625,2024-06-25,25,6,June,2,2024,0
4,20240420,2024-04-20,20,4,April,2,2024,1


In [16]:
# DIM_ROUTE
dim_route = df[['origin', 'destination', 'route', 'route_category']].drop_duplicates().reset_index(drop=True)
dim_route['route_id'] = dim_route.index + 1
dim_route = dim_route[['route_id', 'origin', 'destination', 'route', 'route_category']]

print("dim_route:", dim_route.shape)
dim_route.head()

dim_route: (30, 5)


,route_id,origin,destination,route,route_category
0,1,DXB,ORD,DXB-ORD,Long Haul
1,2,DXB,HYD,DXB-HYD,Medium Haul
2,3,DXB,CDG,DXB-CDG,Long Haul
3,4,DXB,DEL,DXB-DEL,Medium Haul
4,5,DXB,RUH,DXB-RUH,Short Haul


In [17]:
# DIM_AIRCRAFT
dim_aircraft = df[['aircraft_type', 'aircraft_capacity']].drop_duplicates().reset_index(drop=True)
dim_aircraft['aircraft_id'] = dim_aircraft.index + 1
dim_aircraft = dim_aircraft[['aircraft_id', 'aircraft_type', 'aircraft_capacity']]

print("dim_aircraft:", dim_aircraft.shape)
dim_aircraft.head()

dim_aircraft: (6, 3)


,aircraft_id,aircraft_type,aircraft_capacity
0,1,Boeing 777-300ER,396
1,2,Boeing 787-9,296
2,3,Airbus A320,180
3,4,Boeing 737-800,189
4,5,Airbus A350-900,325


In [18]:
# FACT_FLIGHT
df_fact = df.copy()

df_fact['date_id'] = df_fact['flight_date'].dt.strftime('%Y%m%d').astype(int)
df_fact = df_fact.merge(dim_route, on=['origin', 'destination', 'route', 'route_category'], how='left')
df_fact = df_fact.merge(dim_aircraft, on=['aircraft_type', 'aircraft_capacity'], how='left')
df_fact = df_fact.merge(dim_demand_level, on=['demand_level'], how='left')
df_fact = df_fact.merge(dim_season, on=['season'], how='left')

fact_columns = [
    'flight_number', 'date_id', 'route_id', 'aircraft_id', 'demand_level_id', 'season_id',
    'passengers', 'load_factor', 'flight_hours',
    'ticket_revenue', 'ancillary_revenue', 'total_revenue'
] + cost_components_lower + ['total_cost', 'profit', 'profit_margin']

fact_flight = df_fact[fact_columns]

n_before = len(fact_flight)
fact_flight = fact_flight.dropna(subset=["date_id", "route_id", "aircraft_id"])
n_after = len(fact_flight)
print("Jumlah baris fact final:", n_after)
fact_flight.head()

Jumlah baris fact final: 7974


,flight_number,date_id,route_id,aircraft_id,demand_level_id,season_id,passengers,load_factor,flight_hours,ticket_revenue,...,handling_cost,navigation_fees,sales_distribution_cost,passenger_service_cost,overhead_cost,marketing_cost,it_systems_cost,total_cost,profit,profit_margin
0,EK8960,20241220,1,1,1,1,308,0.7791,14.5,410785.49,...,4460.11,10636.90,78193.60,5100.09,88545.73,25883.92,3769.37,506421.76,-42204.33,-0.090915
1,EK3960,20240513,2,2,1,2,234,0.7910,4.2,145890.17,...,5425.84,3027.25,21586.21,6022.20,17057.61,8820.56,1937.81,123323.88,40261.56,0.246119
2,EK7529,20241012,3,2,2,3,251,0.8502,7.5,602841.03,...,3761.85,3431.70,100188.97,6514.22,31347.38,23695.34,3415.99,282266.09,399299.10,0.585856
3,EK4543,20240625,4,2,2,4,229,0.7748,3.5,126485.16,...,4440.93,2079.79,22686.82,4791.21,15328.62,6795.49,2245.61,113919.10,30109.32,0.209051
4,EK3114,20240420,5,3,1,3,142,0.7901,2.2,32651.15,...,3414.18,1662.49,5133.62,3094.65,8252.23,1492.53,1273.27,46580.87,-10058.06,-0.275391


# Load

In [ ]:
db_name = 'airline_data_warehouse.db'
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")

In [20]:
cursor.execute("DROP TABLE IF EXISTS fact_flight;")
cursor.execute("DROP TABLE IF EXISTS dim_date;")
cursor.execute("DROP TABLE IF EXISTS dim_route;")
cursor.execute("DROP TABLE IF EXISTS dim_aircraft;")
cursor.execute("DROP TABLE IF EXISTS dim_demand_level;")
cursor.execute("DROP TABLE IF EXISTS dim_season;")
conn.commit()

In [ ]:
cursor.execute("""
CREATE TABLE dim_date (
    date_id INTEGER PRIMARY KEY,
    flight_date TEXT,
    day INTEGER,
    month INTEGER,
    month_name TEXT,
    quarter INTEGER,
    year INTEGER,
    is_weekend INTEGER
);
""")

cursor.execute("""
CREATE TABLE dim_route (
    route_id INTEGER PRIMARY KEY AUTOINCREMENT,
    origin TEXT,
    destination TEXT,
    route TEXT,
    route_category TEXT
);
""")

cursor.execute("""
CREATE TABLE dim_aircraft (
    aircraft_id INTEGER PRIMARY KEY AUTOINCREMENT,
    aircraft_type TEXT,
    aircraft_capacity INTEGER
);
""")

cursor.execute("""
CREATE TABLE dim_demand_level (
    demand_level_id INTEGER PRIMARY KEY AUTOINCREMENT,
    demand_level TEXT
);
""")

cursor.execute("""
CREATE TABLE dim_season (
    season_id INTEGER PRIMARY KEY AUTOINCREMENT,
    season TEXT
);
""")

cursor.execute("""
CREATE TABLE fact_flight (
    flight_number TEXT PRIMARY KEY,
    date_id INTEGER,
    route_id INTEGER,
    aircraft_id INTEGER,
    demand_level_id INTEGER,
    season_id INTEGER,
    passengers INTEGER,
    load_factor REAL,
    flight_hours REAL,
    ticket_revenue REAL,
    ancillary_revenue REAL,
    total_revenue REAL,
    fuel_cost REAL,
    maintenance_cost REAL,
    crew_cost REAL,
    depreciation_cost REAL,
    insurance_cost REAL,
    airport_fees REAL,
    catering_cost REAL,
    handling_cost REAL,
    navigation_fees REAL,
    sales_distribution_cost REAL,
    passenger_service_cost REAL,
    overhead_cost REAL,
    marketing_cost REAL,
    it_systems_cost REAL,
    total_cost REAL,
    profit REAL,
    profit_margin REAL,
    FOREIGN KEY (date_id) REFERENCES dim_date (date_id),
    FOREIGN KEY (route_id) REFERENCES dim_route (route_id),
    FOREIGN KEY (aircraft_id) REFERENCES dim_aircraft (aircraft_id),
    FOREIGN KEY (demand_level_id) REFERENCES dim_demand_level (demand_level_id),
    FOREIGN KEY (season_id) REFERENCES dim_season (season_id)
);
""")
conn.commit()

In [22]:
dim_date.to_sql('dim_date', conn, if_exists='replace', index=False)
dim_route.to_sql('dim_route', conn, if_exists='replace', index=False)
dim_aircraft.to_sql('dim_aircraft', conn, if_exists='replace', index=False)
dim_demand_level.to_sql('dim_demand_level', conn, if_exists='replace', index=False)
dim_season.to_sql('dim_season', conn, if_exists='replace', index=False)

fact_flight.to_sql('fact_flight', conn, if_exists='replace', index=False)

7974

In [23]:
# Optional Verification Cell
test_schema = pd.read_sql_query(
    "SELECT sql FROM sqlite_master WHERE type='table' AND name='fact_flight';", 
    conn
)
print(test_schema['sql'].iloc[0])

CREATE TABLE "fact_flight" (
"flight_number" TEXT,
  "date_id" INTEGER,
  "route_id" INTEGER,
  "aircraft_id" INTEGER,
  "demand_level_id" INTEGER,
  "season_id" INTEGER,
  "passengers" INTEGER,
  "load_factor" REAL,
  "flight_hours" REAL,
  "ticket_revenue" REAL,
  "ancillary_revenue" REAL,
  "total_revenue" REAL,
  "fuel_cost" REAL,
  "maintenance_cost" REAL,
  "crew_cost" REAL,
  "depreciation_cost" REAL,
  "insurance_cost" REAL,
  "airport_fees" REAL,
  "catering_cost" REAL,
  "handling_cost" REAL,
  "navigation_fees" REAL,
  "sales_distribution_cost" REAL,
  "passenger_service_cost" REAL,
  "overhead_cost" REAL,
  "marketing_cost" REAL,
  "it_systems_cost" REAL,
  "total_cost" REAL,
  "profit" REAL,
  "profit_margin" REAL
)


# Explore DB to Excel

In [24]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('airline_data_warehouse.db')

tables = ['fact_flight', 'dim_date', 'dim_route', 'dim_aircraft', 'dim_season', 'dim_demand_level']

with pd.ExcelWriter('airline_data_warehouse.xlsx', engine='openpyxl') as writer:
    for t in tables:
        df = pd.read_sql(f"SELECT * FROM {t}", conn)
        df.to_excel(writer, sheet_name=t, index=False)

conn.close()